In [ ]:
from groq import Groq
import json

import os 
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

def search_term_meaning(term: str, context: str):
    """
    Search for the meaning of a term given its context using Groq's compound model.
    
    Args:
        term: The term to search for
        context: The context where the term was used
    
    Returns:
        dict: Structured output with one clear meaning of the term
    """
    client = Groq()
    
    prompt = f"""I need help understanding a specific term I encountered.

Term: "{term}"

Context where it was used:
"{context}"

Please search the web and provide ONE clear, concise definition/meaning of this term.
Focus on how it's used in the given context.
Return your answer in this exact JSON format:

{{
    "term": "{term}",
    "definition": "A single clear definition here",
    "context_explanation": "How it applies in the given context",
    "domain": "The field/domain this term belongs to (e.g., Machine Learning, Biology, etc.)"
}}

Only provide the JSON, nothing else."""

    response = client.chat.completions.create(
        model="groq/compound",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    
    # Extract the structured output from the response
    content = response.choices[0].message.content.strip()
    
    # Try to parse JSON from the response
    try:
        # Remove markdown code blocks if present
        if content.startswith("```"):
            content = content.split("```")[1]
            if content.startswith("json"):
                content = content[4:]
        
        structured_data = json.loads(content.strip())
    except json.JSONDecodeError:
        # Fallback if JSON parsing fails
        structured_data = {
            "term": term,
            "definition": content,
            "context_explanation": "See definition above",
            "domain": "General"
        }
    
    # Add metadata
    structured_data["context_provided"] = context
    
    return structured_data


# Example usage
if __name__ == "__main__":
    # Example 1: Technical term
    term1 = "attention mechanism"
    context1 = "The transformer architecture uses a self-attention mechanism to process input sequences in parallel."
    
    print("=" * 80)
    print(f"Searching for: {term1}")
    print("=" * 80)
    
    result1 = search_term_meaning(term1, context1)
    
    print("\n### STRUCTURED OUTPUT ###")
    print(json.dumps(result1, indent=2))
    
    print("\n" + "=" * 80)
    print("\n")
    
    # Example 2: Another term
    term2 = "backpropagation"
    context2 = "During training, the neural network uses backpropagation to update its weights based on the error gradient."
    
    print("=" * 80)
    print(f"Searching for: {term2}")
    print("=" * 80)
    
    result2 = search_term_meaning(term2, context2)
    
    
    print("\n### STRUCTURED OUTPUT ###")
    print(json.dumps(result2, indent=2))
